In [5]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model, load_model, save_model
from tensorflow.keras.layers import Dense, Input, Concatenate
from tensorflow.keras.optimizers import Adam
from collections import deque
import random
import matplotlib.pyplot as plt
import joblib
import os
from datetime import datetime, timedelta

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

In [6]:
class EnergyMaintenanceEnvironment:
    """
    Environment for the energy system maintenance scheduling problem
    """
    def __init__(self, data, maintenance_cost=100, failure_cost=500, 
                maintenance_effect=0.8, degradation_rate=0.05):
        """
        Initialize the environment
        
        Parameters:
        -----------
        data : pandas.DataFrame
            Historical energy data with timestamp, consumption, temperature, etc.
        maintenance_cost : float
            Cost of performing maintenance
        failure_cost : float
            Cost of system failure
        maintenance_effect : float
            Effectiveness of maintenance (how much it improves system condition)
        degradation_rate : float
            Rate at which system condition degrades
        """
        self.data = data.copy()
        
        # Sort by timestamp if not already sorted
        if isinstance(self.data.index, pd.DatetimeIndex):
            self.data = self.data.sort_index()
        else:
            self.data['Timestamp'] = pd.to_datetime(self.data['Timestamp'])
            self.data = self.data.sort_values('Timestamp')
            
        # Add useful features
        self._add_features()
        
        # Parameters
        self.maintenance_cost = maintenance_cost
        self.failure_cost = failure_cost
        self.maintenance_effect = maintenance_effect
        self.degradation_rate = degradation_rate
        
        # State variables
        self.system_condition = 1.0  # Start with perfect condition
        self.time_since_maintenance = 0
        self.current_step = 0
        self.done = False
        
        # Total steps (time periods) in the environment
        self.max_steps = len(self.data)
        
        # Feature scaling
        self.scaler = None
        self._scale_features()
        
    def _add_features(self):
        """Add useful derived features to the data"""
        # Convert timestamp to datetime if it's not already
        if 'Timestamp' in self.data.columns and not pd.api.types.is_datetime64_any_dtype(self.data['Timestamp']):
            self.data['Timestamp'] = pd.to_datetime(self.data['Timestamp'])
        
        # Extract time features
        if 'Timestamp' in self.data.columns:
            self.data['Hour'] = self.data['Timestamp'].dt.hour
            self.data['Day'] = self.data['Timestamp'].dt.day
            self.data['Month'] = self.data['Timestamp'].dt.month
            self.data['DayOfWeek'] = self.data['Timestamp'].dt.dayofweek
            self.data['Weekend'] = (self.data['DayOfWeek'] >= 5).astype(int)
            
        # Calculate rolling statistics for energy consumption if available
        if 'Energy Consumption (kWh)' in self.data.columns:
            self.data['Rolling_Mean_24h'] = self.data['Energy Consumption (kWh)'].rolling(24, min_periods=1).mean()
            self.data['Rolling_Std_24h'] = self.data['Energy Consumption (kWh)'].rolling(24, min_periods=1).std().fillna(0)
            self.data['Rolling_Max_24h'] = self.data['Energy Consumption (kWh)'].rolling(24, min_periods=1).max()
        
        # Calculate load factor (ratio of average to peak demand)
        if 'Rolling_Mean_24h' in self.data.columns and 'Rolling_Max_24h' in self.data.columns:
            self.data['Load_Factor'] = self.data['Rolling_Mean_24h'] / self.data['Rolling_Max_24h'].replace(0, 1)
            
        # Calculate Energy Consumption / Occupancy ratio if both are available
        if 'Energy Consumption (kWh)' in self.data.columns and 'Building Occupancy' in self.data.columns:
            self.data['Energy_per_Occupant'] = self.data['Energy Consumption (kWh)'] / (self.data['Building Occupancy'] + 1)
            
    def _scale_features(self):
        """Scale numerical features for RL state input"""
        from sklearn.preprocessing import StandardScaler
        
        # Select features for the state
        features = [
            'Energy Consumption (kWh)', 'Temperature (°C)', 'Building Occupancy',
            'Hour', 'Day', 'Month', 'DayOfWeek', 'Weekend'
        ]
        
        # Only use features that exist in the data
        self.state_features = [f for f in features if f in self.data.columns]
        
        # Add derived features if they exist
        derived_features = ['Rolling_Mean_24h', 'Rolling_Std_24h', 'Load_Factor', 'Energy_per_Occupant']
        self.state_features.extend([f for f in derived_features if f in self.data.columns])
        
        # Fit scaler
        self.scaler = StandardScaler()
        self.scaler.fit(self.data[self.state_features])
        
    def reset(self):
        """Reset the environment to the initial state"""
        self.system_condition = 1.0
        self.time_since_maintenance = 0
        self.current_step = 0
        self.done = False
        return self._get_state()
    
    def _get_state(self):
        """Get the current state representation"""
        # Get current data point
        current_data = self.data.iloc[self.current_step]
        
        # Extract features
        feature_values = current_data[self.state_features].values.reshape(1, -1)
        
        # Scale features
        scaled_features = self.scaler.transform(feature_values).flatten()
        
        # Combine with system state
        state = np.concatenate([
            scaled_features,
            [self.system_condition],
            [self.time_since_maintenance / 168]  # Normalize time since maintenance (168 hours = 1 week)
        ])
        
        return state
    
    def _calculate_failure_probability(self):
        """Calculate probability of system failure based on condition and usage"""
        # Get current data point
        current_data = self.data.iloc[self.current_step]
        
        # Base probability depends on system condition (lower condition = higher failure probability)
        base_prob = 1 - self.system_condition
        
        # Usage intensity factor
        if 'Energy Consumption (kWh)' in current_data:
            # Normalize consumption by dividing by max value in dataset
            usage_intensity = current_data['Energy Consumption (kWh)'] / self.data['Energy Consumption (kWh)'].max()
            # Higher usage increases failure probability
            usage_factor = 0.5 + (usage_intensity * 0.5)
        else:
            usage_factor = 1.0
            
        # Temperature stress factor
        if 'Temperature (°C)' in current_data:
            # Higher or lower temperatures increase failure probability
            # Calculate deviation from optimal temperature (assume 22°C is optimal)
            temp_deviation = abs(current_data['Temperature (°C)'] - 22) / 40  # Normalize by a max deviation of 40°C
            temp_factor = 0.5 + (temp_deviation * 0.5)
        else:
            temp_factor = 1.0
            
        # Combine factors
        failure_prob = min(0.99, base_prob * usage_factor * temp_factor)
        
        return failure_prob
        
    def step(self, action):
        """
        Take a step in the environment
        
        Parameters:
        -----------
        action : int
            0 = do nothing, 1 = perform maintenance
            
        Returns:
        --------
        next_state : numpy.ndarray
            Next state representation
        reward : float
            Reward for the action
        done : bool
            Whether the episode is done
        info : dict
            Additional information
        """
        # Check if episode is already done
        if self.done:
            return self._get_state(), 0, True, {}
            
        # Get current data
        current_data = self.data.iloc[self.current_step]
        
        # Initialize reward
        reward = 0
        
        # Calculate failure probability before action
        failure_prob = self._calculate_failure_probability()
        
        # Process action
        if action == 1:  # Perform maintenance
            # Cost of maintenance
            reward -= self.maintenance_cost
            
            # Improve system condition
            self.system_condition = min(1.0, self.system_condition + self.maintenance_effect * (1 - self.system_condition))
            
            # Reset time since maintenance
            self.time_since_maintenance = 0
        else:  # Do nothing
            # Check for random failure based on condition
            if np.random.random() < failure_prob:
                # System failure occurred
                reward -= self.failure_cost
                
                # Reset system condition (assume repair after failure)
                self.system_condition = 0.7  # Partial recovery after failure
            
            # Increment time since maintenance
            self.time_since_maintenance += 1
        
        # System degradation (more degradation under higher load)
        if 'Energy Consumption (kWh)' in current_data:
            usage_factor = current_data['Energy Consumption (kWh)'] / self.data['Energy Consumption (kWh)'].max()
            degradation = self.degradation_rate * (0.5 + 0.5 * usage_factor)
        else:
            degradation = self.degradation_rate
            
        self.system_condition = max(0.1, self.system_condition - degradation)
        
        # Move to next step
        self.current_step += 1
        
        # Check if episode is done
        if self.current_step >= self.max_steps - 1:
            self.done = True
            
        # Get next state
        next_state = self._get_state()
        
        # Return step information
        info = {
            'system_condition': self.system_condition,
            'failure_probability': failure_prob,
            'time_since_maintenance': self.time_since_maintenance
        }
        
        return next_state, reward, self.done, info

In [8]:
class DQNAgent:
    """Deep Q-Network Agent for energy system maintenance scheduling"""
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        
        # Hyperparameters
        self.gamma = 0.95  # discount factor
        self.epsilon = 1.0  # exploration rate
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.learning_rate = 0.001
        self.update_target_frequency = 10
        
        # Memory for experience replay
        self.memory = deque(maxlen=10000)
        
        # Main model (for training)
        self.model = self._build_model()
        
        # Target model (for predictions)
        self.target_model = self._build_model()
        self.update_target_model()
        
    def _build_model(self):
        """Build a neural network model for DQN"""
        model = Sequential()
        model.add(Dense(64, input_dim=self.state_size, activation='relu'))
        model.add(Dense(64, activation='relu'))
        model.add(Dense(self.action_size, activation='linear'))
        model.compile(loss='mse', optimizer=Adam(learning_rate=self.learning_rate))
        return model
        
    def update_target_model(self):
        """Update target model with weights from main model"""
        self.target_model.set_weights(self.model.get_weights())
        
    def remember(self, state, action, reward, next_state, done):
        """Store experience in memory"""
        self.memory.append((state, action, reward, next_state, done))
        
    def act(self, state, training=True):
        """Select action based on epsilon-greedy policy"""
        if training and np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        
        act_values = self.model.predict(state.reshape(1, -1), verbose=0)
        return np.argmax(act_values[0])
    
    def replay(self, batch_size):
        """Train model with experience replay"""
        if len(self.memory) < batch_size:
            return
        
        # Sample batch from memory
        minibatch = random.sample(self.memory, batch_size)
        
        for state, action, reward, next_state, done in minibatch:
            target = reward
            if not done:
                target = reward + self.gamma * np.amax(
                    self.target_model.predict(next_state.reshape(1, -1), verbose=0)[0]
                )
            
            target_f = self.model.predict(state.reshape(1, -1), verbose=0)
            target_f[0][action] = target
            self.model.fit(state.reshape(1, -1), target_f, epochs=1, verbose=0)
            
        # Decay epsilon
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
            
    def load(self, name):
        """Load model weights"""
        self.model.load_weights(name)
        
    def save(self, name):
        """Save model weights"""
        self.model.save_weights(name)

class MaintenanceScheduler:
    """
    Wrapper class for predicting and scheduling maintenance using the trained RL agent
    """
    def __init__(self, agent=None, env=None, model_path=None):
        """
        Initialize the scheduler
        
        Parameters:
        -----------
        agent : DQNAgent, optional
            Trained RL agent
        env : EnergyMaintenanceEnvironment, optional
            Environment used for training
        model_path : str, optional
            Path to load saved models from
        """
        if model_path:
            self.load(model_path)
        else:
            self.agent = agent
            self.env = env
    
    def predict_maintenance(self, state_data):
        """
        Predict whether maintenance should be performed
        
        Parameters:
        -----------
        state_data : pd.DataFrame or dict
            Current system state data
            
        Returns:
        --------
        action : int
            0 = no maintenance, 1 = perform maintenance
        """
        # Convert input to appropriate format
        if isinstance(state_data, dict):
            state_data = pd.DataFrame([state_data])
        
        # Ensure we have required features
        for feature in self.env.state_features:
            if feature not in state_data.columns:
                raise ValueError(f"Missing required feature: {feature}")
                
        # Extract features
        feature_values = state_data[self.env.state_features].values.reshape(1, -1)
        
        # Scale features
        scaled_features = self.env.scaler.transform(feature_values).flatten()
        
        # Get system condition and time since maintenance from input or use defaults
        system_condition = state_data.get('system_condition', 1.0)
        if isinstance(system_condition, pd.Series):
            system_condition = system_condition.values[0]
            
        time_since_maintenance = state_data.get('time_since_maintenance', 0)
        if isinstance(time_since_maintenance, pd.Series):
            time_since_maintenance = time_since_maintenance.values[0]
        
        # Combine with system state
        state = np.concatenate([
            scaled_features,
            [system_condition],
            [time_since_maintenance / 168]  # Normalize time since maintenance
        ])
        
        # Get action from agent
        action = self.agent.act(state, training=False)
        
        return action
    
    def schedule_maintenance(self, forecast_data, initial_condition=1.0, initial_time_since_maintenance=0):
        """
        Generate a maintenance schedule for a forecasted period
        
        Parameters:
        -----------
        forecast_data : pd.DataFrame
            Forecasted data for future periods
        initial_condition : float
            Initial system condition
        initial_time_since_maintenance : int
            Initial time since last maintenance
            
        Returns:
        --------
        schedule : pd.DataFrame
            Dataframe with original forecast plus maintenance recommendations
        """
        # Copy forecast data
        schedule = forecast_data.copy()
        
        # Initialize state variables
        system_condition = initial_condition
        time_since_maintenance = initial_time_since_maintenance
        
        # Add columns for maintenance decisions and projected system condition
        schedule['maintenance_recommended'] = 0
        schedule['projected_system_condition'] = np.nan
        schedule['failure_probability'] = np.nan
        
        # Go through forecast period
        for idx in range(len(schedule)):
            # Get current data point
            current_data = schedule.iloc[idx:idx+1].copy()
            
            # Add system state
            current_data['system_condition'] = system_condition
            current_data['time_since_maintenance'] = time_since_maintenance
            
            # Predict maintenance action
            action = self.predict_maintenance(current_data)
            
            # Record decision
            schedule.loc[schedule.index[idx], 'maintenance_recommended'] = action
            schedule.loc[schedule.index[idx], 'projected_system_condition'] = system_condition
            
            # Calculate failure probability
            failure_prob = self._calculate_failure_probability(current_data, system_condition)
            schedule.loc[schedule.index[idx], 'failure_probability'] = failure_prob
            
            # Update state for next period
            if action == 1:  # Perform maintenance
                # Improve system condition
                system_condition = min(1.0, system_condition + self.env.maintenance_effect * (1 - system_condition))
                # Reset time since maintenance
                time_since_maintenance = 0
            else:
                # System degradation
                if 'Energy Consumption (kWh)' in current_data:
                    usage_factor = current_data['Energy Consumption (kWh)'].values[0] / forecast_data['Energy Consumption (kWh)'].max()
                    degradation = self.env.degradation_rate * (0.5 + 0.5 * usage_factor)
                else:
                    degradation = self.env.degradation_rate
                    
                system_condition = max(0.1, system_condition - degradation)
                time_since_maintenance += 1
        
        return schedule
    
    def _calculate_failure_probability(self, data, system_condition):
        """Calculate failure probability based on current data and system condition"""
        if isinstance(data, pd.DataFrame):
            current_data = data.iloc[0]
        else:
            current_data = data
            
        # Base probability depends on system condition
        base_prob = 1 - system_condition
        
        # Usage intensity factor
        if 'Energy Consumption (kWh)' in current_data:
            # Use env's data max for consistency
            usage_intensity = current_data['Energy Consumption (kWh)'] / self.env.data['Energy Consumption (kWh)'].max()
            usage_factor = 0.5 + (usage_intensity * 0.5)
        else:
            usage_factor = 1.0
            
        # Temperature stress factor
        if 'Temperature (°C)' in current_data:
            temp_deviation = abs(current_data['Temperature (°C)'] - 22) / 40
            temp_factor = 0.5 + (temp_deviation * 0.5)
        else:
            temp_factor = 1.0
            
        # Combine factors
        failure_prob = min(0.99, base_prob * usage_factor * temp_factor)
        
        return failure_prob
    
    def save(self, path):
        """Save the model and environment"""
        # Create directory if it doesn't exist
        os.makedirs(path, exist_ok=True)
        
        # Save agent models
        self.agent.model.save(os.path.join(path, 'dqn_model.h5'))
        self.agent.target_model.save(os.path.join(path, 'dqn_target_model.h5'))
        
        # Save environment scaler
        joblib.dump(self.env.scaler, os.path.join(path, 'env_scaler.pkl'))
        
        # Save important environment parameters
        env_params = {
            'maintenance_cost': self.env.maintenance_cost,
            'failure_cost': self.env.failure_cost,
            'maintenance_effect': self.env.maintenance_effect,
            'degradation_rate': self.env.degradation_rate,
            'state_features': self.env.state_features
        }
        joblib.dump(env_params, os.path.join(path, 'env_params.pkl'))
        
        # Save data stats for normalization
        data_stats = {
            'energy_consumption_max': self.env.data['Energy Consumption (kWh)'].max()
            if 'Energy Consumption (kWh)' in self.env.data.columns else None
        }
        joblib.dump(data_stats, os.path.join(path, 'data_stats.pkl'))
    
    def load(self, path):
        """Load the model and environment components"""
        # Load agent models
        self.agent = DQNAgent(0, 2)  # Temporary, will be overwritten
        self.agent.model = load_model(os.path.join(path, 'dqn_model.h5'))
        self.agent.target_model = load_model(os.path.join(path, 'dqn_target_model.h5'))
        
        # Set state size based on model input shape
        self.agent.state_size = self.agent.model.input_shape[1]
        
        # Load environment parameters
        env_params = joblib.load(os.path.join(path, 'env_params.pkl'))
        
        # Create a minimal environment
        self.env = type('MinimalEnvironment', (), {})()
        self.env.maintenance_cost = env_params['maintenance_cost']
        self.env.failure_cost = env_params['failure_cost']
        self.env.maintenance_effect = env_params['maintenance_effect']
        self.env.degradation_rate = env_params['degradation_rate']
        self.env.state_features = env_params['state_features']
        
        # Load environment scaler
        self.env.scaler = joblib.load(os.path.join(path, 'env_scaler.pkl'))
        
        # Load data stats
        self.env.data = type('DataStats', (), {})()
        data_stats = joblib.load(os.path.join(path, 'data_stats.pkl'))
        self.env.data.__dict__.update(data_stats)

def load_and_prepare_data(file_path):
    """Load and prepare data from CSV file"""
    # Load data
    df = pd.read_csv(file_path)
    
    # Convert timestamp to datetime if it's not already
    if 'Timestamp' in df.columns:
        df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    
    # Rename columns if needed
    column_mapping = {
        'Temperature (Â°C)': 'Temperature (°C)',
        'Energy Tariff ($/kWh)': 'Energy Tariff ($/kWh)',
        'Energy Consumption (kWh)': 'Energy Consumption (kWh)',
        'Building Occupancy': 'Building Occupancy'
    }
    
    df = df.rename(columns={k: v for k, v in column_mapping.items() if k in df.columns})
    
    return df

def train_maintenance_rl_agent(data, epochs=5, batch_size=32, output_dir='maintenance_rl_model'):
    """
    Train an RL agent for maintenance scheduling
    
    Parameters:
    -----------
    data : pandas.DataFrame
        Historical energy data
    epochs : int
        Number of training epochs
    batch_size : int
        Batch size for experience replay
    output_dir : str
        Directory to save the trained model
        
    Returns:
    --------
    scheduler : MaintenanceScheduler
        Trained maintenance scheduler
    """
    # Create environment
    env = EnergyMaintenanceEnvironment(
        data, 
        maintenance_cost=50,    # Cost of performing maintenance
        failure_cost=500,       # Cost of system failure
        maintenance_effect=0.4,  # How much maintenance improves system condition
        degradation_rate=0.02   # How fast system degrades per timestep
    )
    
    # Get state size from environment
    state = env.reset()
    state_size = len(state)
    
    # Create agent
    agent = DQNAgent(state_size, 2)  # 2 actions: 0=do nothing, 1=maintenance
    
    # Training variables
    rewards_history = []
    avg_rewards_history = []
    maintenance_history = []
    failure_history = []
    condition_history = []
    
    # Train the agent
    for epoch in range(epochs):
        # Reset environment
        state = env.reset()
        total_reward = 0
        maintenance_count = 0
        failure_count = 0
        
        # Run one episode
        while True:
            # Select action
            action = agent.act(state)
            
            # Take step
            next_state, reward, done, info = env.step(action)
            
            # Store experience
            agent.remember(state, action, reward, next_state, done)
            
            # Update state
            state = next_state
            
            # Count maintenance actions and failures
            if action == 1:
                maintenance_count += 1
            if reward <= -env.failure_cost:
                failure_count += 1
                
            # Add reward
            total_reward += reward
            
            # Record system condition
            condition_history.append(info['system_condition'])
            
            # Train the agent
            if len(agent.memory) > batch_size:
                agent.replay(batch_size)
            
            if done:
                break
        
        # Update target model periodically
        if epoch % agent.update_target_frequency == 0:
            agent.update_target_model()
            
        # Record history
        rewards_history.append(total_reward)
        maintenance_history.append(maintenance_count)
        failure_history.append(failure_count)
        
        # Calculate running average of rewards
        if len(rewards_history) > 10:
            avg_reward = np.mean(rewards_history[-10:])
        else:
            avg_reward = total_reward
        avg_rewards_history.append(avg_reward)
        
        # Print progress
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch + 1}/{epochs}, Avg Reward: {avg_reward:.2f}, "
                  f"Maintenance: {maintenance_count}, Failures: {failure_count}, "
                  f"Epsilon: {agent.epsilon:.2f}")
    
    # Create directory for output
    os.makedirs(output_dir, exist_ok=True)
    
    # Create scheduler
    scheduler = MaintenanceScheduler(agent, env)
    
    # Save the model
    scheduler.save(output_dir)
    
    # Plot training history
    plt.figure(figsize=(15, 10))
    
    plt.subplot(2, 2, 1)
    plt.plot(rewards_history)
    plt.title('Rewards per Episode')
    plt.xlabel('Episode')
    plt.ylabel('Total Reward')
    
    plt.subplot(2, 2, 2)
    plt.plot(avg_rewards_history)
    plt.title('Average Rewards (10 episodes)')
    plt.xlabel('Episode')
    plt.ylabel('Average Reward')
    
    plt.subplot(2, 2, 3)
    plt.plot(maintenance_history)
    plt.title('Maintenance Actions per Episode')
    plt.xlabel('Episode')
    plt.ylabel('Number of Maintenance Actions')
    
    plt.subplot(2, 2, 4)
    plt.plot(failure_history)
    plt.title('Failures per Episode')
    plt.xlabel('Episode')
    plt.ylabel('Number of Failures')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'training_history.png'))
    
    # Plot system condition history
    plt.figure(figsize=(15, 5))
    plt.plot(condition_history)
    plt.title('System Condition over Training')
    plt.xlabel('Step')
    plt.ylabel('System Condition')
    plt.ylim(0, 1)
    plt.savefig(os.path.join(output_dir, 'condition_history.png'))
    
    return scheduler

def evaluate_model(scheduler, test_data, output_dir='maintenance_rl_model'):
    """
    Evaluate trained maintenance scheduler on test data
    
    Parameters:
    -----------
    scheduler : MaintenanceScheduler
        Trained maintenance scheduler
    test_data : pandas.DataFrame
        Test data for evaluation
    output_dir : str
        Directory to save evaluation results
        
    Returns:
    --------
    results : pandas.DataFrame
        Evaluation results
    """
    # Generate maintenance schedule
    schedule = scheduler.schedule_maintenance(test_data)
    
    # Calculate evaluation metrics
    maintenance_rate = schedule['maintenance_recommended'].mean()
    high_risk_periods = (schedule['failure_probability'] > 0.3).sum()
    avg_condition = schedule['projected_system_condition'].mean()
    
    # Print evaluation results
    print("\nModel Evaluation:")
    print(f"Maintenance Rate: {maintenance_rate:.2%}")
    print(f"High Risk Periods: {high_risk_periods} ({high_risk_periods/len(schedule):.2%})")
    print(f"Average System Condition: {avg_condition:.2f}")
    
    # Plot system condition and maintenance recommendations
    plt.figure(figsize=(15, 10))
    
    plt.subplot(3, 1, 1)
    plt.plot(schedule.index, schedule['projected_system_condition'])
    plt.title('Projected System Condition')
    plt.ylabel('Condition')
    plt.ylim(0, 1)
    
    plt.subplot(3, 1, 2)
    plt.plot(schedule.index, schedule['failure_probability'])
    plt.title('Failure Probability')
    plt.ylabel('Probability')
    plt.ylim(0, 1)
    
    plt.subplot(3, 1, 3)
    maintenance_indices = schedule.index[schedule['maintenance_recommended'] == 1]
    plt.vlines(maintenance_indices, 0, 1, colors='r', label='Recommended Maintenance')
    
    if 'Energy Consumption (kWh)' in schedule.columns:
        normalized_consumption = schedule['Energy Consumption (kWh)'] / schedule['Energy Consumption (kWh)'].max()
        plt.plot(schedule.index, normalized_consumption, label='Normalized Energy Consumption')
    
    plt.title('Maintenance Recommendations and Energy Consumption')
    plt.xlabel('Time')
    plt.ylabel('Value')
    plt.legen

In [9]:
# Load and prepare the data
data = load_and_prepare_data(r"C:\Users\91995\OneDrive\Desktop\Sustainathon\DataSets\Updated_Environmental & External Factors.csv")

# Train the RL agent
scheduler = train_maintenance_rl_agent(data, epochs=5, batch_size=32, output_dir='maintenance_rl_model')

# Save the model for future use
scheduler.save('maintenance_rl_model')

# Evaluate the model on test data (optional)
# Assuming you have a separate test dataset
test_data = load_and_prepare_data('test_energy_data.csv')
evaluate_model(scheduler, test_data, output_dir='maintenance_rl_model')

c:\Users\91995\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\91995\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\91995\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\91995\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\91995\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but Standard

KeyboardInterrupt: 